In [ ]:
import xarray as xr
import numpy as np

#### Bedmap3 to 4km

In [ ]:
ds = xr.open_dataset('/home/erwin/data/bedmap/bedmap3.nc')
ds = ds.coarsen(x=8,y=8,boundary='trim').mean()
ds = ds.reindex(y=list(reversed(ds.y)))
ds['x'] = ds.x.astype('double')
ds['y'] = ds.y.astype('double')
ds['surface_topography'] = ds.surface_topography.astype('double')
ds['bed_topography'] = ds.bed_topography.astype('double')
ds['ice_thickness'] = ds.ice_thickness.astype('double')
ds['surface_topography'] = xr.where(np.isnan(ds.surface_topography),0,ds.surface_topography)
ds['ice_thickness'] = xr.where(np.isnan(ds.ice_thickness),0,ds.ice_thickness)

ds = ds.drop_vars(['bed_uncertainty','thickness_survey_count','thickness_uncertainty','mapping'])

ds.to_netcdf('../input/bedmap3_4km.nc')
ds.close()

#### Ocean forcing for calibration

In [ ]:
for run in ['cold','warm']:
    ds = xr.open_dataset(f'/home/erwin/data/ismip7_distr/parameterisations/Ocean_Modelling_Data_old/Mathiot23_{run}_T.nc')
    dsS = xr.open_dataset(f'/home/erwin/data/ismip7_distr/parameterisations/Ocean_Modelling_Data_old/Mathiot23_{run}_S.nc')

    ds['s_an'] = dsS.salinity_ocean

    ds = ds.rename_dims({'z':'depth'})
    ds = ds.rename_vars({'z':'depth','theta_ocean':'t_an'})

    ds['depth'] = -ds.depth
    ds = ds.isel(time=0)
    ds = ds.drop_vars('time')

    ds.to_netcdf(f'../input/Mathiot23_{run}.nc')
    